# ExaMLOps code map — the two sets of code

Everything you see in the dashboard comes from **two distinct sets of code** (the platform ⟂
use-case boundary, ADR 0094):

| Set | What | Who owns it | Editable from here |
|---|---|---|---|
| **1. ExaMLOps plugins** | the *calculations* behind the dashboard — CO₂ / energy / cost, drift, promotion, LLM cost… | the platform, extended by you | **Yes** — author plugins (`plugins/`) |
| **2. Use-case / models / pipelines** | *your* models, their configs, and how they run on **Prefect** (training) and **Ray Serve** (serving) | ExaMLOps users | browse; edit via the pack |

The full platform is mounted read-only at `~/examlops` (browse it in the file panel). Your
authored plugins live in the writable `~/plugins` folder and are shared with the CLI + dashboard.

Run the cells below to list, read, and (for plugins) edit each set.

In [ ]:
import pathlib, examlops
REPO = pathlib.Path(examlops.__file__).resolve().parents[4]   # ~/examlops
print('platform repo:', REPO)
print('examlops package:', pathlib.Path(examlops.__file__).parent)

---
## SET 1 — ExaMLOps plugins (the CO₂ / energy / cost / drift… calculations)

Each dashboard number (carbon, cost, drift score, …) is produced by a swappable **provider**
(ADR 0074). The mechanism lives in `examlops/providers/`; the built-in calculators are the
`*_providers.py` modules; your own plugins live in `~/plugins`.

In [ ]:
# The built-in calculation code (open any of these in the file panel to read the formula):
E = REPO / 'platform/cli/src/examlops'
builtin = {
    'carbon (CO2)':      E / 'finops/carbon_providers.py',
    'cost / energy':     E / 'finops/cost_providers.py',
    'drift':             E / 'drift_providers.py',
    'promotion':         E / 'promotion_providers.py',
    'llm cost/cache/routing/rag': E / 'llmops_providers.py',
    'hpc placement':     E / 'hpc_placement_providers.py',
}
for name, p in builtin.items():
    print(f'{name:32} {p.relative_to(REPO)}')
print('\nprovider engine:', (E / 'providers').relative_to(REPO), '->', sorted(x.name for x in (E/'providers').glob('*.py')))

In [ ]:
# Live list of every calculation plugin the platform can use, by domain (built-in + your authored ones)
import importlib
from examlops.providers import list_providers, default_provider_name
DOMAIN_MODULES = {
    'carbon': 'examlops.finops.carbon_providers', 'cost': 'examlops.finops.cost_providers',
    'drift': 'examlops.drift_providers', 'promotion': 'examlops.promotion_providers',
    'llm_cost': 'examlops.llmops_providers', 'placement': 'examlops.hpc_placement_providers',
}
for d, mod in DOMAIN_MODULES.items():
    importlib.import_module(mod)  # registers the built-ins
    ps = list_providers(d)
    print(f"{d:12} default={default_provider_name(d)!s:18} providers={[p.name for p in ps]}")

In [ ]:
# Read the CO2 formula (or change the path to any other calculator):
print((E / 'finops/carbon_providers.py').read_text()[:1600])

In [ ]:
# Author your OWN calculation plugin (AST-sandboxed, per-project). It is shared with CLI + dashboard.
from examlops.providers import save_provider, set_active_provider, list_project_providers
PROJECT = 'minio-demo'
code = '''
class MyCarbon(Provider):
    name = "my-carbon"
    def metadata(self):
        return ProviderMeta(methodology="kwh = gpu_hours * 0.4 (my grid); co2e_g = kwh * 350", outputs=("kwh","co2e_g"))
    def compute(self, inputs):
        kwh = inputs.get("gpu_hours", 0) * 0.4
        return {"kwh": kwh, "co2e_g": kwh * 350}
'''
# save_provider("carbon", "my-carbon", code, project=PROJECT)  # uncomment to persist
# set_active_provider(PROJECT, "carbon", "my-carbon")          # uncomment to make it the default
print('current plugins for', PROJECT, '->', [(r['domain'], r['name']) for r in list_project_providers(PROJECT)])

---
## SET 2 — Use-case: your models, configs, and pipelines

This is the **user** content (ADR 0094): a *pack* under `usecases/<name>/` names the concrete
models + configs + dataset schemas. The platform runs them through the **Prefect** training engine
(`pipelines/`) and the **Ray Serve** serving engine (`serving/`) — which never name a concrete model.

In [ ]:
# Your use-case pack: models (YAML = single source of truth) + per-model transforms + dataset schemas
PACK = REPO / 'usecases/reference'
print('pack:', PACK.relative_to(REPO))
print('  models:       ', [p.name for p in sorted((PACK/'models').glob('*.yaml'))])
print('  model_configs:', [p.name for p in sorted((PACK/'model_configs').glob('*.py'))])
print('  datasets:     ', [p.name for p in sorted((PACK/'datasets').glob('*'))])
print('\n--- models/jpcp.yaml (a model definition) ---')
print((PACK / 'models/jpcp.yaml').read_text()[:1200])

In [ ]:
# How a model runs: the Prefect training engine and the Ray Serve serving engine
print('Prefect (training) engine  -> pipelines/')
for p in sorted((REPO/'pipelines').glob('*.py')):
    print('   ', p.relative_to(REPO))
print('\nRay Serve (serving) engine -> serving/')
for p in sorted((REPO/'serving').rglob('*.py')):
    print('   ', p.relative_to(REPO))

---
## Dashboard → code map

| Dashboard item | Set | Code |
|---|---|---|
| FinOps CO₂ / carbon | 1 | `examlops/finops/carbon_providers.py` (+ your `~/plugins/<proj>/carbon/*.py`) |
| FinOps cost / energy | 1 | `examlops/finops/cost_providers.py` (+ `~/plugins/<proj>/cost/*.py`) |
| Drift score / actions | 1 | `examlops/drift_providers.py`, `examlops/data/drift.py` |
| LLMOps cost / cache / routing | 1 | `examlops/llmops_providers.py` |
| Models / MLOps registry | 2 | `usecases/reference/models/*.yaml`, `pipelines/model_loader.py` |
| Pipelines (training) | 2 | `pipelines/pipeline_generator.py`, `pipelines/usecase.py` |
| Serving / traffic split | 2 | `serving/ray_serving/app.py` |
| Projects / workbenches | platform | `platform/services/dashboard`, `examlops/workbenches` |

**To edit set 1** (a calculation): author a plugin above → it's live everywhere. **To edit set 2**
(a model/pipeline): edit the pack YAML/config under `~/examlops/usecases/reference/` and deploy with
`exa pipeline deploy` / `exa serve reload`.